In [24]:
import os
from dotenv import load_dotenv

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

print("project root:", PROJECT_ROOT)

project root: C:\Users\Anshul\Agentic_RAG


In [25]:
from typing import TypedDict

class AgentState(TypedDict):
    messages:     list
    query:        str
    retry_count:  int
    final_answer: str

In [26]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_core.tools import tool
from tavily import TavilyClient

# reconnect to chromadb
client_db = chromadb.PersistentClient(path=os.path.join(PROJECT_ROOT, "chroma_db"))
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client_db.get_collection(name="ai_safety_papers", embedding_function=embed_fn)

# reconnect to tavily
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def retrieve(query, top_k=5):
    results = collection.query(query_texts=[query], n_results=top_k, include=["documents", "metadatas", "distances"])
    chunks = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        chunks.append({"text": doc, "title": meta.get("title", ""), "authors": meta.get("authors", ""), "year": meta.get("year", ""), "page_num": meta.get("page_num", ""), "score": round(1 - dist, 4)})
    return chunks

def format_chunks(chunks):
    if not chunks:
        return "No relevant chunks found."
    parts = []
    for i, c in enumerate(chunks, 1):
        parts.append(f"[Chunk {i} | {c['title']} | {c['authors']}, {c['year']} | Page {c['page_num']} | Score: {c['score']}]\n{c['text']}")
    return "\n\n---\n\n".join(parts)

def format_web_results(response):
    parts = []
    if response.get("answer"):
        parts.append(f"[Quick Answer]\n{response['answer']}")
    for i, r in enumerate(response["results"], 1):
        parts.append(f"[Web Result {i} | {r['title']} | {r['url']}]\n{r['content']}")
    return "\n\n---\n\n".join(parts)

@tool
def rag_search(query: str) -> str:
    """
    Search the AI safety research knowledge base containing 6 foundational papers.
    Use this tool when the question is about:
      - Core AI safety concepts (reward hacking, scalable oversight, RLHF)
      - Transformer and attention architecture
      - Constitutional AI methodology
      - Anything that would be in a research paper published before 2023
    Do NOT use this for current events, recent news, or anything after 2023.
    """
    return format_chunks(retrieve(query))

@tool
def web_search_tool(query: str) -> str:
    """
    Search the live web for current information about AI safety and related topics.
    Use this tool when the question is about:
      - Recent news, announcements or events from 2024 onwards
      - Current status of AI policy or regulations
      - Recent statements by researchers or companies
      - Anything NOT likely to be in a research paper published before 2023
    Do NOT use this for foundational concepts already covered in research papers.
    """
    response = tavily.search(query=query, search_depth="basic", max_results=5, include_answer=True)
    return format_web_results(response)

TOOLS = [rag_search, web_search_tool]
print("tools ready:", [t.name for t in TOOLS])

tools ready: ['rag_search', 'web_search_tool']


In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = 'gemini-2.0-flash-lite',
    temperature=0,
    google_api_key=os.getenv('GEMINI_API_KEY')
).bind_tools(TOOLS)

print('llm ready')

llm ready


### Node 1 - Planner

In [28]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

PLANNER_PROMPT = """You are an AI safety research assistant with access to two tools:

1. rag_search — searches 6 foundational AI safety papers (pre-2023)
   USE FOR: core concepts, methodologies, paper-specific questions

2. web_search_tool — searches the live web
   USE FOR: recent news, current events, policy updates, anything after 2023

Analyze the question and call the appropriate tool(s) with specific search queries.
If the question has both a conceptual and current-events aspect, call both tools.
"""


def planner_node(state: AgentState) -> AgentState:
    messages = [SystemMessage(content = PLANNER_PROMPT)] + state['messages']
    response = llm.invoke(messages)

    # log what the agent decided to call
    if response.tool_calls:
        for tc in response.tool_calls:
            icon = 'RS' if tc["name"] == "rag_search" else "WS"
            print(f"  {icon} calling {tc['name']}: {tc['args'].get('query', '')!r}")
    else:
        print("no tool calls made")

    return {"messages": state["messages"] + [response]}

### Node 2 - Tools

In [29]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(TOOLS)

### Node 3 - Reflect

In [30]:
import json
from langchain_google_genai import ChatGoogleGenerativeAI

reflect_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    temperature=0,
    google_api_key=os.getenv("GEMINI_API_KEY")
)

REFLECT_PROMPT = """You are evaluating whether retrieved information is sufficient to answer the user's question.

Respond with ONLY valid JSON in this exact format, no extra text:
{
  "sufficient": true or false,
  "gaps": "what is still missing, or empty string if sufficient"
}
"""

def reflect_node(state: AgentState) -> AgentState:
    tool_results = [msg.content for msg in state['message'] if isinstance(msg, ToolMessage)]

    reflection_prompt = f"""
    Question: {state['query']}

    Information gathered:
    {chr(10).join(tool_results)}

    Is this sufficient to answer the question?
    """

    response = reflect_llm.invoke([
        SystemMessage(content=REFLECT_PROMPT),
        HumanMessage(content=reflection_prompt)
    ])

    try:
        verdict = json.loads(response.content)
        sufficient = verdict.get('sufficient', True)
        gaps = verdict.get('gaps', '')
        print(f"  sufficient: {sufficient}" + (f" | gap: {gaps}" if gaps else ""))
    except json.JSONDecodeError:
        sufficient = True
        gaps = ""
        print("  could not parse reflection — proceeding to synthesis")

        if not sufficient and state['retry_count']<2:
            retry_message = HumanMessage(
                content=f'Still missing: {gaps}. Please search for this.'
            )

        return {
            'messages': state['messages'] + [retry_message],
            'retry_count': state['retry_count'] + 1
        }

    return {}
    

### Node 4 - Synthesizer

In [31]:
synth_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    temperature=0.2,
    google_api_key=os.getenv("GEMINI_API_KEY")
)

SYNTH_PROMPT = """You are a research assistant. Synthesize the retrieved information into a clear, 
well-structured answer. Cite sources — for RAG results use (Author et al., Year), 
for web results use the URL. Be honest about uncertainty."""

def synthesizer_node(state: AgentState) -> AgentState:
    tool_results = [msg.content for msg in state['messages'] if isinstance(msg, ToolMessage)]
    prompt = f"""
        Question: {state['query']}
        
        Retrieved information:
        {'='*50}
        {chr(10).join(tool_results)}
        {'='*50}
        
        Write a thorough, cited answer.
        """
    response = synth_llm.invoke([
            SystemMessage(content=SYNTH_PROMPT),
            HumanMessage(content=prompt)
        ])
        
    print("  answer synthesized ✓")
    return {"final_answer": response.content}

In [32]:
from langgraph.graph import StateGraph, END

def after_planner(state: AgentState) -> str:
    last_msg = state['messages'][-1]
    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
        return 'tools'

    return 'synthesize'

def after_reflect(state: AgentState) -> str:
    last_msg = state['messages'][-1]
    if isinstance(last_msg, HumanMessage) and 'Still Missing' in last_msg.content:
        return 'planner'

    return 'synthesize'


graph = StateGraph(AgentState)

graph.add_node('planner', planner_node)
graph.add_node('tools', tool_node)
graph.add_node('reflect', reflect_node)
graph.add_node('synthesize', synthesizer_node)

graph.set_entry_point('planner')

graph.add_conditional_edges('planner', after_planner, {
    'tools': 'tools',
    'synthesize': 'synthesize'
})

graph.add_edge('tools', 'reflect')
graph.add_conditional_edges('reflect', after_reflect, {
    'planner': 'planner',
    'synthesize': 'synthesize'
})

graph.add_edge('synthesize', END)

app = graph.compile()
print('graph compiled')

graph compiled


In [35]:
def run_query(question: str) -> str:
    initial_state = {
        'messages': [HumanMessage(content=question)],
        'query': question,
        'retry_count': 0,
        'final_anwer': ''
    }

    result = app.invoke(initial_state)
    return result['final_answer']

print("=" * 50)
print("QUERY 1: foundational concept")
print("=" * 50)
answer = run_query("What is reward hacking and why is it a problem?")
print("\nFINAL ANSWER:")
print(answer)

QUERY 1: foundational concept


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 12.973490235s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemi

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)